# Instant-1.7B — Step 3: LoRA fine-tune (answer directly, no thinking)

**Goal:** teach `LiquidAI/LFM3-1.7B` to answer **directly** when thinking is OFF: no hidden reasoning, short answers, but the accuracy of its own best answers.

**Data:** the 2,069 verified examples from Step 2 (`results/step2_data`): *question → the model's own shortest correct answer, thinking removed*. They are loaded straight from GitHub, so nothing needs to be uploaded.

**How it works:**

| Step | What happens | GPU |
|---|---|---|
| 1. Before | The **original** model answers the 103 validation questions with thinking OFF (in the background) | GPU 1 |
| 2. Train | LoRA (small add-on weights, ~1% of the model) learns the direct answers. Every prompt is built with the model's own chat template and `enable_thinking=False`, exactly as in real use. Only the answer tokens are trained on | GPU 0 |
| 3. After | The **fine-tuned** model answers the same 103 questions | GPU 0 |
| 4. Compare | Hidden reasoning, cut-off answers, length, and a rough correctness check, before vs after | – |
| 5. Save | LoRA adapter (small) + merged full model, ready for Step 4 (full benchmark) and Hugging Face | – |

The real test comes in Step 4: the full benchmark notebook (HumanEval+, MBPP+, LiveCodeBench, GSM8K, IFEval, IFBench, BFCL) on the fine-tuned model. This notebook's check is only a quick sanity check.

**Kaggle:** Accelerator **GPU T4 x2**, Internet **On**. Rough time: 40–60 min (training ~25–35 min, checks ~10 min each, merge ~3 min).

## 1. Install

In [ ]:
!pip uninstall -y -q torchao      # Kaggle's old torchao breaks peft
!pip install -q -U transformers accelerate peft

## 2. Settings
The defaults are a safe first run. If the model still reasons in secret afterwards, try `EPOCHS = 3`; if answers get worse or repetitive, try `LR = 5e-5`.

In [ ]:
import os
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

MODEL = "LiquidAI/LFM3-1.7B"
DATA_URL = "https://raw.githubusercontent.com/Sanskar1724/Instant-1.7b_off-Thinking/main/results/step2_data"

# LoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",   # attention layers
                  "in_proj", "out_proj",                     # short-convolution layers
                  "w1", "w2", "w3"]                          # feed-forward layers

# Training
EPOCHS = 2
LR = 1e-4
WARMUP_STEPS = 8
TOKENS_PER_BATCH = 8192     # padded tokens per micro-batch (lower it if you see "out of memory" often)
ACCUM = 2                   # micro-batches per optimizer step
MAX_LEN = 3072              # longer examples are skipped
EVAL_EVERY = 20             # optimizer steps between validation-loss checks
SEED = 0

# Before/after check
CHECK_MAX_NEW = 1024        # token cap per answer in the quick check
CHECK_BATCH = 16

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
DATA_DIR = os.path.join(WORK, "step2_data")
OUT_DIR = os.path.join(WORK, "step3")
ADAPTER_DIR = os.path.join(OUT_DIR, "lora_adapter")
MERGED_DIR = os.path.join(WORK, "instant-1.7b-merged")
for d in (DATA_DIR, OUT_DIR):
    os.makedirs(d, exist_ok=True)
print("Working dir:", WORK)

## 3. GPUs, tokenizer, data
Uses `train.jsonl` / `val.jsonl` from an attached Kaggle dataset if there is one, otherwise downloads them from GitHub.

In [ ]:
import glob, json, shutil, sys, time, subprocess, urllib.request, random, math
import torch
from transformers import AutoTokenizer

assert torch.cuda.is_available(), "No GPU: set Accelerator to 'GPU T4 x2'."
N_GPUS = torch.cuda.device_count()
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(N_GPUS)])
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16   # T4: float16
print("dtype:", DTYPE)

for name in ("train.jsonl", "val.jsonl"):
    dest = os.path.join(DATA_DIR, name)
    if os.path.exists(dest):
        continue
    found = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if found:
        shutil.copy(found[0], dest); print("using", found[0])
    else:
        urllib.request.urlretrieve(f"{DATA_URL}/{name}", dest); print("downloaded", name)

def read_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

TRAIN_RAW = read_jsonl(os.path.join(DATA_DIR, "train.jsonl"))
VAL_RAW = read_jsonl(os.path.join(DATA_DIR, "val.jsonl"))
print(f"train: {len(TRAIN_RAW)}  val: {len(VAL_RAW)}")

tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
PAD = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
IM_END = tokenizer.convert_tokens_to_ids("<|im_end|>")
assert IM_END is not None and IM_END != tokenizer.unk_token_id, "chat end token not found"
print("pad id:", PAD, " <|im_end|> id:", IM_END)

## 4. Build training examples
Each example = the **prompt exactly as the model sees it with thinking OFF** (chat template, tools included, ending in the empty `<think></think>`) + the **answer** + `<|im_end|>`.
The loss is computed **only on the answer** (the prompt tokens are masked), so the model learns *what to answer*, not to repeat the question.

In [ ]:
def render_prompt(messages, tools=None):
    return tokenizer.apply_chat_template(messages, tools=tools, tokenize=False, add_generation_prompt=True,
                                         enable_thinking=False)

def encode(rec):
    prompt = render_prompt(rec["messages"][:-1], rec.get("tools"))
    p_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    a_ids = tokenizer(rec["messages"][-1]["content"], add_special_tokens=False)["input_ids"] + [IM_END]
    return {"id": rec["id"], "domain": rec["domain"], "input_ids": p_ids + a_ids,
            "labels": [-100] * len(p_ids) + a_ids, "len": len(p_ids) + len(a_ids), "n_target": len(a_ids),
            "prompt_ids": p_ids}

TRAIN = [encode(r) for r in TRAIN_RAW]
VAL = [encode(r) for r in VAL_RAW]
n0 = len(TRAIN)
TRAIN = [x for x in TRAIN if x["len"] <= MAX_LEN]
VAL_LOSS_SET = [x for x in VAL if x["len"] <= MAX_LEN]
print(f"skipped {n0 - len(TRAIN)} training examples longer than {MAX_LEN} tokens")

import pandas as pd
dl = pd.DataFrame([{"domain": x["domain"], "tokens": x["len"], "answer_tokens": x["n_target"]} for x in TRAIN])
display(dl.groupby("domain").agg(examples=("tokens", "size"), mean_tokens=("tokens", "mean"), max_tokens=("tokens", "max"),
                                 mean_answer_tokens=("answer_tokens", "mean")).round(0))
print(f"training tokens per epoch: {dl.tokens.sum():,} (answer tokens: {dl.answer_tokens.sum():,})")

# Show one tool example exactly as trained (end of prompt + answer)
ex_raw = next((r for r in TRAIN_RAW if r["domain"] == "tools"), TRAIN_RAW[0])
ex = encode(ex_raw)
print("\n===== end of prompt =====\n" + tokenizer.decode(ex["prompt_ids"][-80:]))
print("===== trained answer =====\n" + tokenizer.decode([t for t in ex["labels"] if t != -100]))
assert tokenizer.decode(ex["prompt_ids"]).rstrip().endswith("<think></think>"), "prompt should end with the empty think block"

## 5. Quick-check worker
A small script that answers the validation questions with thinking OFF and saves every answer at once. It runs as a separate process, so the *original* model can answer on GPU 1 **while** GPU 0 trains.

In [ ]:
%%writefile check_worker.py
import argparse, json, os, time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

ap = argparse.ArgumentParser()
ap.add_argument("--gpu", type=int, required=True)
ap.add_argument("--model", required=True)
ap.add_argument("--adapter", default="")
ap.add_argument("--prompts", required=True)
ap.add_argument("--out", required=True)
ap.add_argument("--max_new", type=int, default=1024)
ap.add_argument("--batch", type=int, default=16)
args = ap.parse_args()

def log(*a):
    print(time.strftime("%H:%M:%S"), f"[GPU{args.gpu}]", *a, flush=True)

dev = torch.device(f"cuda:{args.gpu}")
torch.cuda.set_device(dev)
dtype = torch.bfloat16 if torch.cuda.get_device_capability(dev)[0] >= 8 else torch.float16
tok = AutoTokenizer.from_pretrained(args.model, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(args.model, dtype=dtype, device_map={"": args.gpu}, trust_remote_code=True)
if args.adapter:
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, args.adapter).merge_and_unload()
model.eval()
PAD = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
_eos = model.generation_config.eos_token_id
EOS = set(_eos if isinstance(_eos, list) else [_eos])
EOS.add(tok.convert_tokens_to_ids("<|im_end|>"))
EOS.discard(None)
DROP = [t for t in tok.all_special_tokens if t not in ("<think>", "</think>")]
log("model loaded", "with adapter" if args.adapter else "(original)")

rows = [json.loads(l) for l in open(args.prompts)]
done = set()
if os.path.exists(args.out):
    for l in open(args.out):
        try: done.add(json.loads(l)["id"])
        except Exception: pass
todo = sorted([r for r in rows if r["id"] not in done], key=lambda r: -len(r["prompt_ids"]))
out = open(args.out, "a")
bs, i = args.batch, 0
while i < len(todo):
    chunk = todo[i:i + bs]
    L = max(len(r["prompt_ids"]) for r in chunk)
    ids = torch.full((len(chunk), L), PAD, dtype=torch.long)
    mask = torch.zeros((len(chunk), L), dtype=torch.long)
    for k, r in enumerate(chunk):
        ids[k, L - len(r["prompt_ids"]):] = torch.tensor(r["prompt_ids"])
        mask[k, L - len(r["prompt_ids"]):] = 1
    t0 = time.perf_counter()
    try:
        with torch.no_grad():
            gen = model.generate(input_ids=ids.to(dev), attention_mask=mask.to(dev), max_new_tokens=args.max_new,
                                 do_sample=False, repetition_penalty=1.1, pad_token_id=PAD, eos_token_id=sorted(EOS))
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if bs == 1:
            raise
        bs = max(1, bs // 2); log("out of memory -> batch", bs); continue
    secs = time.perf_counter() - t0
    for k, r in enumerate(chunk):
        toks = gen[k, L:].tolist()
        stop = next((j for j, t in enumerate(toks) if t in EOS), None)
        toks = toks if stop is None else toks[:stop]
        text = tok.decode(toks, skip_special_tokens=False)
        for t in DROP:
            text = text.replace(t, "")
        out.write(json.dumps({"id": r["id"], "text": text, "n_tokens": len(toks), "truncated": stop is None,
                              "seconds": secs / len(chunk)}) + "\n")
    out.flush()
    i += len(chunk)
    log(f"{i}/{len(todo)} answers ({secs:.0f}s for {len(chunk)})")
out.close()
log("all done")

In [ ]:
CHECK_DIR = os.path.join(OUT_DIR, "check")
os.makedirs(CHECK_DIR, exist_ok=True)
PROMPTS_FILE = os.path.join(CHECK_DIR, "val_prompts.jsonl")
with open(PROMPTS_FILE, "w") as f:
    f.writelines(json.dumps({"id": x["id"], "prompt_ids": x["prompt_ids"]}) + "\n" for x in VAL)

def start_check(name, gpu, adapter=""):
    out = os.path.join(CHECK_DIR, f"{name}.jsonl")
    cmd = [sys.executable, os.path.abspath("check_worker.py"), "--gpu", str(gpu), "--model", MODEL, "--prompts", PROMPTS_FILE,
           "--out", out, "--max_new", str(CHECK_MAX_NEW), "--batch", str(CHECK_BATCH)] + (["--adapter", adapter] if adapter else [])
    log_f = open(os.path.join(CHECK_DIR, f"{name}.log"), "a")
    return subprocess.Popen(cmd, stdout=log_f, stderr=subprocess.STDOUT)

def n_lines(p):
    return sum(1 for _ in open(p)) if os.path.exists(p) else 0

def wait_check(name, proc):
    out = os.path.join(CHECK_DIR, f"{name}.jsonl")
    while proc.poll() is None:
        print(f"\r{name}: {n_lines(out)}/{len(VAL)} answers", end="", flush=True); time.sleep(15)
    print(f"\r{name}: {n_lines(out)}/{len(VAL)} answers")
    if proc.returncode != 0:
        print(open(os.path.join(CHECK_DIR, f"{name}.log")).read()[-3000:])
        raise RuntimeError(f"{name} check failed (log above). Run the cell again to continue.")

# The original model answers on GPU 1 while GPU 0 trains (with one GPU it runs after training instead).
BASE_PROC = None
if N_GPUS > 1 and n_lines(os.path.join(CHECK_DIR, "original_off.jsonl")) < len(VAL):
    BASE_PROC = start_check("original_off", gpu=1)
    print("original model check started on GPU 1 (runs in the background)")

## 6. Load the model and add LoRA
The original weights stay frozen (float16 on T4). Only the small LoRA matrices are trained, in float32 for stability. Gradient checkpointing saves memory.

In [ ]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

random.seed(SEED); torch.manual_seed(SEED)
DEV = torch.device("cuda:0")
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=DTYPE, device_map={"": 0}, trust_remote_code=True)
model.config.use_cache = False
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model = get_peft_model(model, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                                         target_modules=TARGET_MODULES, task_type="CAUSAL_LM"))
for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.float()
model.print_trainable_parameters()
INNER = model.get_base_model()          # Lfm3ForCausalLM with LoRA layers inside
print("LoRA layers:", sum(1 for n, _ in model.named_modules() if n.endswith("lora_A")))

## 7. Train
Examples of similar length are batched together (less padding = faster). To save memory, the vocabulary layer is only run on the **answer** positions, in chunks. The table updates live; the adapter with the lowest validation loss is kept.

In [ ]:
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from IPython.display import display, HTML

def make_batches(rows, seed):
    rng = random.Random(seed)
    order = sorted(range(len(rows)), key=lambda i: rows[i]["len"] + rng.random() * 64)
    batches, cur, mx = [], [], 0
    for i in order:
        L = rows[i]["len"]
        if cur and max(mx, L) * (len(cur) + 1) > TOKENS_PER_BATCH:
            batches.append(cur); cur, mx = [], 0
        cur.append(i); mx = max(mx, L)
    if cur: batches.append(cur)
    rng.shuffle(batches)
    return batches

def collate(rows):
    L = max(r["len"] for r in rows)
    ids = torch.full((len(rows), L), PAD, dtype=torch.long)
    lab = torch.full((len(rows), L), -100, dtype=torch.long)
    mask = torch.zeros((len(rows), L), dtype=torch.long)
    for k, r in enumerate(rows):
        ids[k, :r["len"]] = torch.tensor(r["input_ids"]); lab[k, :r["len"]] = torch.tensor(r["labels"]); mask[k, :r["len"]] = 1
    return ids.to(DEV), mask.to(DEV), lab.to(DEV)

def _ce(h, y):
    with torch.autocast("cuda", dtype=DTYPE):
        logits = INNER.lm_head(h)
    return F.cross_entropy(logits.float(), y, reduction="sum")

def loss_sum(rows):
    # summed cross-entropy over answer tokens only
    ids, mask, lab = collate(rows)
    with torch.autocast("cuda", dtype=DTYPE):
        h = INNER.model(input_ids=ids, attention_mask=mask, use_cache=False).last_hidden_state
    y = lab[:, 1:]
    sel = y != -100
    h, y = h[:, :-1][sel], y[sel]
    total = 0.0
    for i in range(0, len(y), 2048):
        if torch.is_grad_enabled():
            total = total + checkpoint(_ce, h[i:i + 2048], y[i:i + 2048], use_reentrant=False)
        else:
            total = total + _ce(h[i:i + 2048], y[i:i + 2048])
    return total

@torch.no_grad()
def val_loss():
    model.eval()
    tot, n = 0.0, 0
    for b in make_batches(VAL_LOSS_SET, 0):
        rows = [VAL_LOSS_SET[i] for i in b]
        tot += float(loss_sum(rows)); n += sum(r["n_target"] for r in rows)
    model.train()
    return tot / n

def backward_rows(rows, denom):
    # forward+backward; on "out of memory" split the batch in half and retry
    try:
        loss = loss_sum(rows)
        scaler.scale(loss / denom).backward()
        return float(loss)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(rows) == 1:
            print("\nskipped one example (out of memory)"); return 0.0
        half = len(rows) // 2
        return backward_rows(rows[:half], denom) + backward_rows(rows[half:], denom)

params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.0, betas=(0.9, 0.999))
scaler = torch.amp.GradScaler("cuda", enabled=(DTYPE == torch.float16))
steps_per_epoch = math.ceil(len(make_batches(TRAIN, 0)) / ACCUM)
TOTAL_STEPS = steps_per_epoch * EPOCHS
def lr_at(step):   # linear warm-up, then cosine down to 10%
    if step < WARMUP_STEPS: return LR * (step + 1) / WARMUP_STEPS
    t = (step - WARMUP_STEPS) / max(1, TOTAL_STEPS - WARMUP_STEPS)
    return LR * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * t)))
print(f"{len(TRAIN)} examples, {EPOCHS} epochs, {TOTAL_STEPS} optimizer steps")

LOG = []
v0 = val_loss(); best = v0
LOG.append({"step": 0, "epoch": 0, "train_loss": None, "val_loss": v0, "lr": 0, "minutes": 0})
print(f"validation loss before training: {v0:.4f}")
status = display(HTML("<pre>starting...</pre>"), display_id=True)
model.train()
step, t0, tok_seen = 0, time.time(), 0
for epoch in range(EPOCHS):
    batches = make_batches(TRAIN, SEED + epoch)
    for g in range(0, len(batches), ACCUM):
        group = [[TRAIN[i] for i in b] for b in batches[g:g + ACCUM]]
        denom = sum(r["n_target"] for rows in group for r in rows)
        for gr in opt.param_groups: gr["lr"] = lr_at(step)
        tot = sum(backward_rows(rows, denom) for rows in group)
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        step += 1
        tok_seen += sum(r["len"] for rows in group for r in rows)
        rec = {"step": step, "epoch": epoch + (g + ACCUM) / len(batches), "train_loss": tot / denom, "val_loss": None,
               "lr": lr_at(step - 1), "minutes": (time.time() - t0) / 60}
        if step % EVAL_EVERY == 0 or step == TOTAL_STEPS:
            rec["val_loss"] = val_loss()
            if rec["val_loss"] < best:
                best = rec["val_loss"]; model.save_pretrained(ADAPTER_DIR)
        LOG.append(rec)
        el = time.time() - t0
        last_val = next((r["val_loss"] for r in reversed(LOG) if r["val_loss"] is not None), v0)
        status.update(HTML(f"<pre>step {step}/{TOTAL_STEPS} | epoch {rec['epoch']:.2f} | train loss {rec['train_loss']:.4f} | "
                           f"val loss {last_val:.4f} (best {best:.4f}) | {tok_seen / el:.0f} tokens/s | "
                           f"{el / 60:.1f} min, ~{el / step * (TOTAL_STEPS - step) / 60:.0f} min left | "
                           f"GPU0 memory {torch.cuda.max_memory_allocated(0) / 2**30:.1f} GB</pre>"))
if not os.path.exists(os.path.join(ADAPTER_DIR, "adapter_config.json")):
    model.save_pretrained(ADAPTER_DIR)     # validation loss never improved: keep the final adapter
pd.DataFrame(LOG).to_csv(os.path.join(OUT_DIR, "train_log.csv"), index=False)
print(f"done in {(time.time() - t0) / 60:.1f} min | validation loss {v0:.4f} -> best {best:.4f} | adapter saved to {ADAPTER_DIR}")

In [ ]:
import matplotlib.pyplot as plt
lg = pd.DataFrame(LOG)
fig, ax = plt.subplots(figsize=(8, 3.5))
t = lg[lg.train_loss.notna()]
ax.plot(t.step, t.train_loss, alpha=0.4, label="train loss (per step)")
ax.plot(t.step, t.train_loss.rolling(10, min_periods=1).mean(), label="train loss (smoothed)")
v = lg[lg.val_loss.notna()]
ax.plot(v.step, v.val_loss, "o-", label="validation loss")
ax.set_xlabel("optimizer step"); ax.set_ylabel("loss (answer tokens)"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "loss.png"), dpi=120); plt.show()

## 8. Before vs after (quick check on the 103 validation questions)
Both models answer with thinking **OFF**, the same settings (greedy, repetition penalty 1.1), and a cap of `CHECK_MAX_NEW` tokens.

In [ ]:
import gc
del model, INNER, opt, params, scaler
gc.collect(); torch.cuda.empty_cache()

tuned_proc = start_check("finetuned_off", gpu=0, adapter=ADAPTER_DIR)
if BASE_PROC is not None:
    wait_check("original_off", BASE_PROC)
wait_check("finetuned_off", tuned_proc)
if n_lines(os.path.join(CHECK_DIR, "original_off.jsonl")) < len(VAL):      # one-GPU case
    wait_check("original_off", start_check("original_off", gpu=0))

In [ ]:
import re
REF = {r["id"]: r for r in VAL_RAW}

def answer_part(text):
    return text.split("</think>")[-1].strip()

def last_number(s):
    m = re.findall(r"-?\d[\d,]*\.?\d*", s.replace("$", ""))
    if not m: return None
    try: return float(m[-1].replace(",", "").rstrip("."))
    except ValueError: return None

def calls(s):
    out = []
    for f in re.finditer(r"<function=([^>\s]+)>(.*?)</function>", s, re.S):
        ps = sorted((p.group(1), " ".join(p.group(2).split())) for p in re.finditer(r"<parameter=([^>\s]+)>(.*?)</parameter>", f.group(2), re.S))
        out.append(json.dumps([f.group(1), ps]))
    return sorted(out)

def agrees(rec, text):
    # rough: does the answer agree with the verified answer? (only where that can be checked simply)
    a, ref = answer_part(text), rec["messages"][-1]["content"]
    d = rec["domain"]
    if d == "gsm8k":
        x, y = last_number(a), last_number(ref); return None if y is None else (x is not None and abs(x - y) < 1e-6)
    if d == "tools":  return calls(a) == calls(ref)
    if d == "no_tool": return "<tool_call>" not in a
    return None

rows = []
for name in ("original_off", "finetuned_off"):
    _seen = set()
    for r in read_jsonl(os.path.join(CHECK_DIR, f"{name}.jsonl")):
        if r["id"] in _seen: continue          # a re-run cell can write an answer twice
        _seen.add(r["id"]); rec = REF[r["id"]]
        rows.append({"model": name, "id": r["id"], "domain": rec["domain"], "tokens": r["n_tokens"], "cut_off": r["truncated"],
                     "hidden_reasoning": ("</think>" in r["text"]) or ("<think>" in r["text"]),
                     "agrees": agrees(rec, r["text"]), "seconds": r["seconds"], "text": r["text"]})
CH = pd.DataFrame(rows)

def summary(df):
    return pd.Series({"answers": len(df), "hidden_reasoning": df.hidden_reasoning.mean(), "cut_off": df.cut_off.mean(),
                      "mean_tokens": df.tokens.mean(), "median_tokens": df.tokens.median(),
                      "agrees_with_verified": df.agrees.dropna().astype(float).mean() if df.agrees.notna().any() else float("nan")})

tab = CH.groupby(["domain", "model"]).apply(summary).unstack("model")
overall = CH.groupby("model").apply(summary).T
fmt = {c: "{:.0%}" for c in tab.columns if c[0] in ("hidden_reasoning", "cut_off", "agrees_with_verified")}
print("Overall (thinking OFF, 103 validation questions):")
display(overall.style.format("{:.2f}"))
print("By domain:")
display(tab.style.format(fmt, na_rep="–").format({c: "{:.0f}" for c in tab.columns if "tokens" in c[0]}))
print("'agrees_with_verified' = same final number (math), same tool calls (tools), no call when none is needed (no_tool).\n"
      "Code and instruction answers are only checked properly in Step 4 (full benchmark).")
CH.drop(columns="text").to_csv(os.path.join(CHECK_DIR, "check_summary.csv"), index=False)

In [ ]:
# Side by side: a few answers before and after
for d in ["mbpp", "gsm8k", "if", "tools", "apps", "no_tool"]:
    ids = [i for i in CH[(CH.domain == d)].id.unique()][:1]
    for i in ids:
        q = REF[i]["messages"][-2]["content"]
        print(f"=================== {d}: {q[:300]!r}")
        for name in ("original_off", "finetuned_off"):
            r = CH[(CH.id == i) & (CH.model == name)].iloc[0]
            print(f"--- {name} ({r.tokens} tokens{', CUT OFF' if r.cut_off else ''}{', HIDDEN REASONING' if r.hidden_reasoning else ''}):")
            print(r.text[:700])
        print()

## 9. Merge and save
- `step3/lora_adapter/`: the LoRA adapter (small)
- `instant-1.7b-merged/`: full model with the adapter merged in, plus Liquid's model code and licence files (needed for Step 4 and for Hugging Face)
- `step3_results.zip`: adapter + training log + loss chart + check results (Output panel → download). The merged model (~3.4 GB) is not in the zip.

A quick load test of the merged folder runs at the end.

In [ ]:
from peft import PeftModel
from huggingface_hub import snapshot_download

base = AutoModelForCausalLM.from_pretrained(MODEL, dtype=DTYPE, device_map={"": 0}, trust_remote_code=True)
merged = PeftModel.from_pretrained(base, ADAPTER_DIR).merge_and_unload()
merged.config.use_cache = True
merged.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)

# Liquid's model code (trust_remote_code) and licence/notice files must travel with the weights
src = snapshot_download(MODEL, allow_patterns=["*.py", "LICENSE*", "NOTICE*", "generation_config.json", "*.jinja"])
for f in os.listdir(src):
    if f.endswith(".py") or f.startswith(("LICENSE", "NOTICE")) or (f in ("generation_config.json", "chat_template.jinja")
                                                                     and not os.path.exists(os.path.join(MERGED_DIR, f))):
        shutil.copy(os.path.join(src, f), os.path.join(MERGED_DIR, f))
with open(os.path.join(MERGED_DIR, "MODIFICATIONS.md"), "w") as f:
    f.write("# Modifications\n\nThese weights were modified from LiquidAI/LFM3-1.7B by LoRA fine-tuning "
            f"(r={LORA_R}, alpha={LORA_ALPHA}, {EPOCHS} epochs, lr={LR}) on the model's own verified direct answers, "
            "then merged. The model code files are unchanged copies of Liquid's. "
            "Licence: LFM Open License v1.0 (see LICENSE). Not affiliated with Liquid AI.\n")
del base, merged; gc.collect(); torch.cuda.empty_cache()
print("merged model saved:", MERGED_DIR)
print(sorted(os.listdir(MERGED_DIR)))

# load test: the merged folder must work on its own
m2 = AutoModelForCausalLM.from_pretrained(MERGED_DIR, dtype=DTYPE, device_map={"": 0}, trust_remote_code=True).eval()
t2 = AutoTokenizer.from_pretrained(MERGED_DIR, trust_remote_code=True)
p = t2.apply_chat_template([{"role": "user", "content": "Write a Python function that returns the square of a number."}],
                           tokenize=False, add_generation_prompt=True, enable_thinking=False)
ids = t2(p, return_tensors="pt", add_special_tokens=False).to("cuda:0")
with torch.no_grad():
    o = m2.generate(**ids, max_new_tokens=200, do_sample=False, repetition_penalty=1.1)
print("load test answer:\n", t2.decode(o[0, ids["input_ids"].shape[1]:], skip_special_tokens=True))
del m2; gc.collect(); torch.cuda.empty_cache()

In [ ]:
json.dump(dict(model=MODEL, lora=dict(r=LORA_R, alpha=LORA_ALPHA, dropout=LORA_DROPOUT, target_modules=TARGET_MODULES),
               epochs=EPOCHS, lr=LR, warmup_steps=WARMUP_STEPS, tokens_per_batch=TOKENS_PER_BATCH, accum=ACCUM,
               max_len=MAX_LEN, seed=SEED, train_examples=len(TRAIN), val_examples=len(VAL),
               val_loss_before=LOG[0]["val_loss"], val_loss_best=min(r["val_loss"] for r in LOG if r["val_loss"] is not None),
               check=overall.to_dict()),
          open(os.path.join(OUT_DIR, "run_info.json"), "w"), indent=1, default=str)
shutil.make_archive(os.path.join(WORK, "step3_results"), "zip", OUT_DIR)
print("saved", os.path.join(WORK, "step3_results.zip"))

PUSH_TO_HUB = False          # True: upload the merged model as a PRIVATE Hugging Face repo (needs Kaggle secret HF_TOKEN)
HUB_REPO = "Sanskar1724/Instant-1.7B-preview"
if PUSH_TO_HUB:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import HfApi
    api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
    api.create_repo(HUB_REPO, private=True, exist_ok=True)
    api.upload_folder(folder_path=MERGED_DIR, repo_id=HUB_REPO)
    api.upload_folder(folder_path=ADAPTER_DIR, repo_id=HUB_REPO, path_in_repo="lora_adapter")
    print("uploaded to", HUB_REPO, "(private)")